In [1]:
import json
import math
import time
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("/Users/dc/geha/finetuning/data/test")

CHUNKS_PATH = DATA_DIR / "chunks.jsonl"
QA_PATH = DATA_DIR / "qa_pairs.filtered-existing.jsonl"

BASE_MODEL = "BAAI/bge-base-en-v1.5"
FINETUNED_MODEL = Path(
    "/Users/dc/geha/finetuning/models/geha-dental-retriever"
)

QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

W1019 11:47:09.248000 30015 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
def load_jsonl(path):
    with Path(path).open() as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


chunk_rows = load_jsonl(CHUNKS_PATH)
qa_rows = load_jsonl(QA_PATH)

corpus_ids = [row["chunk_id"] for row in chunk_rows]

corpus_texts = [
    row.get("contextualized_text") or row["text"]
    for row in chunk_rows
]

corpus_id_set = set(corpus_ids)

for row in qa_rows:
    missing = set(row["source_chunk_ids"]) - corpus_id_set
    assert not missing, (
        f"Question {row['id']} references missing chunks: {missing}"
    )

print("Corpus chunks:", len(corpus_texts))
print("Test questions:", len(qa_rows))

Corpus chunks: 167
Test questions: 41


In [3]:
def evaluate_retriever(model_name_or_path, qa_rows, k=10):
    model = SentenceTransformer(str(model_name_or_path))

    # Corpus encoding is an indexing operation, so it is not included
    # in online query latency.
    corpus_embeddings = model.encode(
        corpus_texts,
        batch_size=32,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    recall_at_5 = []
    reciprocal_ranks = []
    ndcg_at_10 = []
    latencies_ms = []

    for row in qa_rows:
        query = QUERY_PREFIX + row["question"]
        relevant_ids = set(row["source_chunk_ids"])

        start = time.perf_counter()

        query_embedding = model.encode(
            query,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        # With normalized embeddings, inner product equals cosine similarity.
        scores = query_embedding @ corpus_embeddings.T
        ranked_indices = np.argsort(-scores)[:k]

        latency_ms = (time.perf_counter() - start) * 1000
        latencies_ms.append(latency_ms)

        ranked_ids = [corpus_ids[i] for i in ranked_indices]

        # Recall@5
        retrieved_at_5 = set(ranked_ids[:5])
        recall_at_5.append(
            len(retrieved_at_5 & relevant_ids) / len(relevant_ids)
        )

        # MRR@10
        reciprocal_rank = 0.0
        for rank, chunk_id in enumerate(ranked_ids[:10], start=1):
            if chunk_id in relevant_ids:
                reciprocal_rank = 1.0 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

        # Binary-relevance nDCG@10
        dcg = sum(
            1.0 / math.log2(rank + 1)
            for rank, chunk_id in enumerate(ranked_ids[:10], start=1)
            if chunk_id in relevant_ids
        )

        ideal_hits = min(len(relevant_ids), 10)
        ideal_dcg = sum(
            1.0 / math.log2(rank + 1)
            for rank in range(1, ideal_hits + 1)
        )

        ndcg_at_10.append(dcg / ideal_dcg if ideal_dcg else 0.0)

    return {
        "model": str(model_name_or_path),
        "questions": len(qa_rows),
        "recall@5": float(np.mean(recall_at_5)),
        "mrr@10": float(np.mean(reciprocal_ranks)),
        "ndcg@10": float(np.mean(ndcg_at_10)),
        "latency_median_ms": float(np.median(latencies_ms)),
        "latency_p95_ms": float(np.percentile(latencies_ms, 95)),
    }

In [4]:
baseline_results = evaluate_retriever(
    BASE_MODEL,
    qa_rows,
)

print(json.dumps(baseline_results, indent=2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "model": "BAAI/bge-base-en-v1.5",
  "questions": 41,
  "recall@5": 0.6463414634146342,
  "mrr@10": 0.7261130468447541,
  "ndcg@10": 0.6774334194266803,
  "latency_median_ms": 12.961333151906729,
  "latency_p95_ms": 14.490375062450767
}


In [5]:
results_dir = Path("/Users/dc/geha/finetuning/results")
results_dir.mkdir(parents=True, exist_ok=True)

with (results_dir / "baseline.json").open("w") as file:
    json.dump(baseline_results, file, indent=2)

In [6]:
# Run this cell once if validation/qa_pairs.jsonl does not exist.
# It uses the finetuning virtual environment directly; activating it in a
# separate Jupyter shell command is unnecessary.
import subprocess

GEHA_ROOT = Path("/Users/dc/geha")
VALIDATION_QA_PATH = (
    GEHA_ROOT / "finetuning/data/validation/qa_pairs.jsonl"
)
RUN_VALIDATION_GENERATION = False

if VALIDATION_QA_PATH.exists():
    print(f"Using existing validation QA: {VALIDATION_QA_PATH}")
elif RUN_VALIDATION_GENERATION:
    command = " ".join([
        "source ~/.zshrc &&",
        "/Users/dc/geha/finetuning/.venv/bin/python",
        "-m finetuning.generate_ragas_qa",
        "--input /Users/dc/geha/finetuning/data/validation/chunks.jsonl",
        "--output-dir /Users/dc/geha/finetuning/data/validation",
        "--testset-size 20",
        "--model gpt-4.1-mini",
        "--max-workers 1",
        "--max-retries 15",
    ])
    subprocess.run(
        ["zsh", "-lc", command],
        cwd=GEHA_ROOT,
        check=True,
    )
else:
    print(
        "Validation QA is missing. Set RUN_VALIDATION_GENERATION=True "
        "and run this cell again."
    )

Using existing validation QA: /Users/dc/geha/finetuning/data/validation/qa_pairs.jsonl


In [7]:
# Dependencies, including accelerate, are installed from requirements.txt.


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [8]:
from sentence_transformers import InputExample, evaluation, losses
from torch.utils.data import DataLoader

TRAIN_DIR = Path("/Users/dc/geha/finetuning/data/train")
VALIDATION_DIR = Path("/Users/dc/geha/finetuning/data/validation")

assert (TRAIN_DIR / "qa_pairs.jsonl").exists(), "Training QA is missing"
assert (VALIDATION_DIR / "qa_pairs.jsonl").exists(), (
    "Validation QA is missing. Run the validation-generation cell first."
)

train_chunk_rows = load_jsonl(TRAIN_DIR / "chunks.jsonl")
validation_chunk_rows = load_jsonl(VALIDATION_DIR / "chunks.jsonl")
training_rows = load_jsonl(TRAIN_DIR / "qa_pairs.jsonl")
validation_rows = load_jsonl(VALIDATION_DIR / "qa_pairs.jsonl")

retrieval_chunk_rows = train_chunk_rows + validation_chunk_rows
retrieval_corpus = {
    row["chunk_id"]: row.get("contextualized_text") or row["text"]
    for row in retrieval_chunk_rows
}

train_examples = []
for row in training_rows:
    for chunk_id in row["source_chunk_ids"]:
        if chunk_id not in retrieval_corpus:
            raise KeyError(f"Unknown training chunk: {chunk_id}")
        train_examples.append(
            InputExample(
                texts=[
                    QUERY_PREFIX + row["question"],
                    retrieval_corpus[chunk_id],
                ]
            )
        )

train_loader = DataLoader(
    train_examples,
    batch_size=16,
    shuffle=True,
    drop_last=False,
)

validation_queries = {
    row["id"]: QUERY_PREFIX + row["question"]
    for row in validation_rows
}
validation_relevant_docs = {
    row["id"]: set(row["source_chunk_ids"])
    for row in validation_rows
}

validation_evaluator = evaluation.InformationRetrievalEvaluator(
    queries=validation_queries,
    corpus=retrieval_corpus,
    relevant_docs=validation_relevant_docs,
    name="validation",
    accuracy_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    mrr_at_k=[10],
    ndcg_at_k=[10],
    map_at_k=[10],
    show_progress_bar=True,
)

training_model = SentenceTransformer(BASE_MODEL)
validation_before_training = validation_evaluator(training_model)
print("Validation before training:")
print(json.dumps(validation_before_training, indent=2))

train_loss = losses.MultipleNegativesRankingLoss(training_model)
training_model.fit(
    train_objectives=[(train_loader, train_loss)],
    evaluator=validation_evaluator,
    epochs=3,
    warmup_steps=max(1, len(train_loader) // 10),
    evaluation_steps=max(1, len(train_loader)),
    output_path=str(FINETUNED_MODEL),
    save_best_model=True,
    show_progress_bar=True,
)

validation_after_training = validation_evaluator(training_model)
print("Validation after training:")
print(json.dumps(validation_after_training, indent=2))
print(f"Saved best encoder to: {FINETUNED_MODEL}")

/var/folders/17/_mc3jc653kx0ddt49myjkj480000gn/T/ipykernel_30015/3520792160.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import InputExample, evaluation, losses
/var/folders/17/_mc3jc653kx0ddt49myjkj480000gn/T/ipykernel_30015/3520792160.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, evaluation, losses


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks: 100%|██| 1/1 [00:01<00:00,  1.18s/it]

Validation after training:
{
  "validation_cosine_accuracy@1": 0.7916666666666666,
  "validation_cosine_accuracy@3": 0.9583333333333334,
  "validation_cosine_accuracy@5": 1.0,
  "validation_cosine_accuracy@10": 1.0,
  "validation_cosine_precision@1": 0.7916666666666666,
  "validation_cosine_precision@3": 0.4444444444444444,
  "validation_cosine_precision@5": 0.30833333333333335,
  "validation_cosine_precision@10": 0.15833333333333333,
  "validation_cosine_recall@1": 0.5138888888888888,
  "validation_cosine_recall@3": 0.7777777777777777,
  "validation_cosine_recall@5": 0.8680555555555555,
  "validation_cosine_recall@10": 0.8888888888888888,
  "validation_cosine_ndcg@10": 0.8182280730904198,
  "validation_cosine_mrr@10": 0.8784722222222222,
  "validation_cosine_map@10": 0.7440972222222221
}
Saved best encoder to: /Users/dc/geha/finetuning/models/geha-dental-retriever


In [9]:
assert FINETUNED_MODEL.exists(), (
    f"Fine-tuned model not found: {FINETUNED_MODEL}"
)

finetuned_results = evaluate_retriever(
    FINETUNED_MODEL,
    qa_rows,
)

print(json.dumps(finetuned_results, indent=2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "model": "/Users/dc/geha/finetuning/models/geha-dental-retriever",
  "questions": 41,
  "recall@5": 0.6869918699186991,
  "mrr@10": 0.7271680216802169,
  "ndcg@10": 0.7000945347552451,
  "latency_median_ms": 14.712166041135788,
  "latency_p95_ms": 16.004167031496763
}


In [10]:
comparison = {
    metric: {
        "baseline": baseline_results[metric],
        "finetuned": finetuned_results[metric],
        "absolute_change": (
            finetuned_results[metric]
            - baseline_results[metric]
        ),
    }
    for metric in ["recall@5", "mrr@10", "ndcg@10"]
}

print(json.dumps(comparison, indent=2))

{
  "recall@5": {
    "baseline": 0.6463414634146342,
    "finetuned": 0.6869918699186991,
    "absolute_change": 0.04065040650406493
  },
  "mrr@10": {
    "baseline": 0.7261130468447541,
    "finetuned": 0.7271680216802169,
    "absolute_change": 0.0010549748354627964
  },
  "ndcg@10": {
    "baseline": 0.6774334194266803,
    "finetuned": 0.7000945347552451,
    "absolute_change": 0.02266111532856474
  }
}
